In [ ]:

# ==================================================
# 5CLOUD - VELIB DATA PLATFORM
# Notebook Azure Synapse : Bronze -> Silver -> Gold
# ==================================================


# Paramètre transmis par Azure Data Factory
batch_id = ""

=== 5CLOUD - AZURE SYNAPSE ===
Batch : 0610bc1b-5d65-462b-8a84-335af2737c68
Bronze : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/bronze/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Silver : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/silver/velib/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Global : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/global/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Gold Communes : abfss://velib-data@stvelibchifa2026.dfs.core.windows.net/gold/velib/communes/batch_id=0610bc1b-5d65-462b-8a84-335af2737c68
Configuration Synapse prête !


In [ ]:
# 5CLOUD - Configuration Azure Synapse

if not batch_id or not batch_id.strip():
    raise ValueError("Le paramètre batch_id doit être fourni par ADF.")

BATCH_ID = batch_id.strip()

STORAGE_ACCOUNT = "stvelibchifa2026"
CONTAINER = "velib-data"

ADLS_ROOT = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
)

BRONZE_PATH = (
    f"{ADLS_ROOT}/bronze/velib/batch_id={BATCH_ID}"
)

SILVER_PATH = (
    f"{ADLS_ROOT}/silver/velib/batch_id={BATCH_ID}"
)

GOLD_GLOBAL_PATH = (
    f"{ADLS_ROOT}/gold/velib/global/batch_id={BATCH_ID}"
)

GOLD_COMMUNES_PATH = (
    f"{ADLS_ROOT}/gold/velib/communes/batch_id={BATCH_ID}"
)

print("=== 5CLOUD - AZURE SYNAPSE ===")
print("Batch :", BATCH_ID)
print("Bronze :", BRONZE_PATH)
print("Silver :", SILVER_PATH)
print("Gold Global :", GOLD_GLOBAL_PATH)
print("Gold Communes :", GOLD_COMMUNES_PATH)
print("Configuration Synapse prête !")

In [ ]:
print("=== SYNAPSE : CONFIGURATION MANIFESTE BRONZE ===")

# Chemin du manifeste associé au lot
MANIFEST_PATH = f"{BRONZE_PATH}/manifest.json"

print("Manifeste Bronze :", MANIFEST_PATH)

# Lecture à effectuer dans le workspace Synapse
def load_collection_time(spark, manifest_path):
    import json
    from datetime import datetime

    # Lecture du manifeste depuis ADLS
    manifest_rows = spark.read.text(
        manifest_path,
        wholetext=True
    ).collect()

    manifest = json.loads(
        manifest_rows[0]["value"]
    )

    collection_time = datetime.fromisoformat(
        manifest["manifest_written_at_utc"].replace(
            "Z", "+00:00"
        )
    )

    return collection_time


def load_bronze_manifest(spark, manifest_path, batch_id):
    import json

    rows = spark.read.text(
        manifest_path,
        wholetext=True
    ).collect()

    if len(rows) != 1:
        raise ValueError("Manifeste Bronze absent ou invalide")

    manifest = json.loads(rows[0]["value"])

    # Vérifier le batch transmis par ADF
    if manifest.get("batch_id") != batch_id:
        raise ValueError(
            "Batch ID du manifeste différent du batch ADF"
        )

    # Vérifier le statut de la collecte
    if manifest.get("collection_status") != "copy_completed":
        raise ValueError("Collecte Bronze non terminée")

    expected_pages = int(manifest["expected_page_count"])
    expected_stations = int(manifest["expected_station_count"])

    if expected_pages <= 0 or expected_stations <= 0:
        raise ValueError("Volumes Bronze attendus invalides")

    print("Manifeste Bronze vérifié")
    print("Pages attendues :", expected_pages)
    print("Stations attendues :", expected_stations)

    return manifest



print("Fonction de lecture du manifeste préparée !")


In [ ]:
from pyspark.sql import functions as F

print("=== SYNAPSE : LECTURE BRONZE ===")

# Chemin des fichiers JSON de Chifa
BRONZE_PAGES_PATH = f"{BRONZE_PATH}/pages/page_*.json"

# Lecture des 16 pages JSON dans Azure
pages_df = (
    spark.read
    .option("multiLine", "true")
    .json(BRONZE_PAGES_PATH)
)

# Les stations se trouvent dans la clé "results"
stations_df = (
    pages_df
    .select(F.explode("results").alias("station"))
    .select("station.*")
)

print("Stations Bronze :", stations_df.count())
print("Nombre de colonnes :", len(stations_df.columns))

stations_df.select(
    "stationcode",
    "name",
    "capacity",
    "numbikesavailable",
    "numdocksavailable"
).show(5, truncate=False)

assert stations_df.count() > 0


# ==========================================
# VALIDATION DE LA COMPLETUDE BRONZE
# ==========================================

manifest = load_bronze_manifest(
    spark, MANIFEST_PATH, BATCH_ID
)

expected_pages = int(manifest["expected_page_count"])
expected_stations = int(manifest["expected_station_count"])

# Compter les fichiers JSON réellement lus
actual_pages = (
    pages_df
    .select(F.input_file_name().alias("file_path"))
    .distinct()
    .count()
)

# Compter les stations réellement collectées
actual_stations = stations_df.count()

print("Pages attendues :", expected_pages)
print("Pages collectées :", actual_pages)
print("Stations attendues :", expected_stations)
print("Stations collectées :", actual_stations)


from src.platform_project.bronze_validation import (
    validate_bronze_manifest
)

validate_bronze_manifest(
    manifest=manifest,
    batch_id=BATCH_ID,
    actual_pages=actual_pages,
    actual_stations=actual_stations
)

print("Validation du manifeste et des volumes Bronze réussie !")


# Vérifier que les pages vont de 0000 à N-1
import re

page_paths = [
    row["file_path"]
    for row in pages_df.select(
        F.input_file_name().alias("file_path")
    ).distinct().collect()
]

page_numbers = []
for path in page_paths:
    match = re.search(r"/page_(\d+)\.json$", path)
    if not match:
        raise ValueError(f"Nom de page invalide : {path}")
    page_numbers.append(int(match.group(1)))

assert sorted(page_numbers) == list(range(expected_pages)), (
    "Pages Bronze manquantes ou numérotation incorrecte"
)

# Vérifier les doublons de stations
duplicate_stations = (
    stations_df
    .groupBy("stationcode")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_stations == 0, (
    f"Bronze contient {duplicate_stations} codes station dupliqués"
)

print("Numérotation des pages et unicité des stations validées !")
print("Lecture Bronze Azure réussie !")



In [ ]:
import json
from datetime import datetime

from src.platform_project.cleaning import clean_velib_data
from src.platform_project.quality import check_data_quality
from src.platform_project.silver_history import prepare_silver_history

print("=== SYNAPSE : TRANSFORMATION SILVER ===")

# Valeur de référence du manifeste Bronze
# À remplacer ensuite par la lecture du manifeste ADLS
#collection_time = datetime.fromisoformat(
#    "2026-10-28T21:35:06.283930+00:00"
#)

collection_time = load_collection_time(
    spark,
    MANIFEST_PATH
)

print("Date du manifeste :", collection_time)

# Nettoyage Silver
silver_df = clean_velib_data(
    stations_df,
    BATCH_ID,
    collection_time
)

# Contrôle qualité
valid_df, rejected_df = check_data_quality(silver_df)

# Déduplication
silver_history_df = prepare_silver_history(valid_df)

print("Stations Silver :", silver_df.count())
print("Stations valides :", valid_df.count())
print("Stations rejetées :", rejected_df.count())
print("Stations après déduplication :", silver_history_df.count())

print("Transformation Silver Synapse terminée !")


In [ ]:

from src.platform_project.aggregations import (
    build_gold_global,
    build_gold_communes
)

print("=== SYNAPSE : AGRÉGATIONS GOLD ===")

# Calcul des KPI globaux
gold_global_df = build_gold_global(silver_history_df)

# Calcul des KPI par commune
gold_communes_df = build_gold_communes(silver_history_df)

print("Nombre de lignes Gold Global :", gold_global_df.count())
print("Nombre de communes :", gold_communes_df.count())

print("\n--- KPI GLOBAUX ---")
gold_global_df.show(truncate=False)

print("\n--- TOP 10 COMMUNES ---")
gold_communes_df.orderBy(
    F.desc("total_bikes")
).show(10, truncate=False)

print("Agrégations Gold Synapse terminées !")


In [ ]:

print("=== CONTRÔLES AVANT ÉCRITURE AZURE ===")

silver_count = silver_history_df.count()
gold_global_count = gold_global_df.count()
gold_communes_count = gold_communes_df.count()

print("Silver :", silver_count)
print("Gold Global :", gold_global_count)
print("Gold Communes :", gold_communes_count)

assert silver_count > 0, "Aucune station Silver valide"
assert gold_global_count == 1, "Gold Global doit contenir une ligne"
assert gold_communes_count > 0, "Aucune commune Gold"

kpi = gold_global_df.first()

assert kpi is not None
assert kpi["total_stations"] == silver_count

assert (
    kpi["mechanical_bikes"] + kpi["electric_bikes"]
    == kpi["total_bikes"]
), "Incohérence dans le total des vélos"

unique_count = silver_history_df.select(
    "batch_id", "station_id"
).distinct().count()

assert unique_count == silver_count, "Doublons Silver détectés"

print("Contrôles dynamiques réussis !")


In [ ]:
print("=== SYNAPSE : ÉCRITURE SILVER & GOLD ===")

# Sécurité : ne pas lancer les écritures
# tant que l'infrastructure n'est pas prête.
ENABLE_AZURE_WRITE = True

if ENABLE_AZURE_WRITE:

    from src.platform_project.azure_storage import (
        write_parquet_to_adls,
        read_parquet_from_adls
    )

    # Vérifier les résultats avant publication
    assert silver_history_df.count() > 0
    assert gold_global_df.count() == 1
    assert gold_communes_df.count() > 0

    # Enregistrer Silver
    write_parquet_to_adls(
        silver_history_df,
        SILVER_PATH
    )

    # Enregistrer Gold Global
    write_parquet_to_adls(
        gold_global_df,
        GOLD_GLOBAL_PATH
    )

    # Enregistrer Gold Communes
    write_parquet_to_adls(
        gold_communes_df,
        GOLD_COMMUNES_PATH
    )

    # Vérifier les fichiers publiés
    saved_silver = read_parquet_from_adls(
        spark, SILVER_PATH
    )

    saved_global = read_parquet_from_adls(
        spark, GOLD_GLOBAL_PATH
    )

    saved_communes = read_parquet_from_adls(
        spark, GOLD_COMMUNES_PATH
    )

    assert saved_silver.count() == silver_history_df.count()
    assert saved_global.count() == 1
    assert saved_communes.count() == gold_communes_df.count()

    print("Écriture et relecture Azure réussies !")

else:
    print("Écriture Azure désactivée.")
    print("En attente du workspace et des accès Synapse.")


=== SYNAPSE : ÉCRITURE SILVER & GOLD ===
Écriture Azure désactivée.
En attente du workspace et des accès Synapse.


In [ ]:
# ==========================================
# 5CLOUD - PUBLICATION DU LOT GOLD
# ==========================================

from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

print("=== PUBLICATION DU LOT GOLD ===")

# Cette cellule doit s'exécuter uniquement après
# les contrôles et les relectures Parquet réussis.

assert silver_count > 0
assert gold_global_count == 1
assert gold_communes_count > 0

publication_path = (
    f"{ADLS_ROOT}/gold/velib/publications/"
    f"batch_id={BATCH_ID}"
)

publication_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("status", StringType(), False),
    StructField("published_at_utc", TimestampType(), False)
])

publication_df = spark.createDataFrame(
    [(
        BATCH_ID,
        "success",
        datetime.now(timezone.utc)
    )],
    schema=publication_schema
)


# Vérifier les données réellement enregistrées dans ADLS
saved_silver = spark.read.parquet(SILVER_PATH)
saved_global = spark.read.parquet(GOLD_GLOBAL_PATH)
saved_communes = spark.read.parquet(GOLD_COMMUNES_PATH)

assert saved_silver.count() == silver_count
assert saved_global.count() == gold_global_count
assert saved_communes.count() == gold_communes_count

# Vérifier que tous les fichiers appartiennent au même lot
for df in [saved_silver, saved_global, saved_communes]:
    assert df.filter(
        F.col("batch_id").isNull() |
        (F.col("batch_id") != BATCH_ID)
    ).count() == 0, "Batch ID incohérent dans les fichiers Parquet"


print("Validation du lot avant publication réussie !")

publication_df.write.mode("errorifexists").parquet(
    publication_path
)

print("Lot Gold publié :", BATCH_ID)
print("Registre :", publication_path)
